In [6]:
import pyspark.sql. functions as F 
from pyspark. sql import SparkSession 
from pyspark.sql.window import Window

In [7]:
jar_files = ["/home/jovyan/jars/postgresql-42.7.9.jar",
            "/home/jovyan/jars/clickhouse-jdbc-0.9.8-all.jar"]

In [8]:
spark = (
    SparkSession
    .builder
    .appName("SparkWeb")
    .config("spark.jars", ",".join(jar_files))
    .getOrCreate()
)

In [9]:
",".join(jar_files)

'/home/jovyan/jars/postgresql-42.7.9.jar,/home/jovyan/jars/clickhouse-jdbc-0.9.8-all.jar'

In [10]:
spark

In [6]:
df_pg = (
    spark.read
    .format("jdbc")
    .option("url", "jdbc:postgresql://postgres:5432/postgres")
    .option("dbtable", "(select 1 as test_value) as t")
    .option("user", "postgres")
    .option("password", "postgres")
    .option("driver", "org.postgresql.Driver")
    .load()
)

df_pg.show()

+----------+
|test_value|
+----------+
|         1|
+----------+



In [7]:
df_ch = (
    spark.read
    .format("jdbc")
    .option("url", "jdbc:clickhouse://clickhouse:8123/default")
    .option("dbtable", "(select 1 as test_value) as t")
    .option("user", "clickhouse")
    .option("password", "clickhouse")
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
    .load()
)

df_ch.show()

+----------+
|test_value|
+----------+
|         1|
+----------+



In [ ]:
#CSV

In [11]:
path = '/home/jovyan/data/'

In [72]:
campaigns_dict = (
    spark.read
    .option('header', True)
    .csv(f'{path}campaigns_dict.csv')
)

In [13]:
campaigns_dict.show(5, truncate=False)

+-----------+------------------------------------------+
|campaign_id|campaign_name                             |
+-----------+------------------------------------------+
|1          |year_modern_kitchen_launch_20250115       |
|2          |quarter_custom_kitchens_showcase_20240210 |
|3          |month_smart_kitchen_promotion_20240305    |
|4          |year_luxury_kitchens_exhibit_20240420     |
|5          |quarter_ecofriendly_kitchen_offer_20240512|
+-----------+------------------------------------------+
only showing top 5 rows


In [107]:
ph_host = 'postgres_spark'
pg_port = '5432'
pg_db = 'postgres'
pg_table = 'costs'
pg_user = 'postgres'
pg_pass = 'postgres'

In [15]:
df = (spark.read
     .option('header', True)
     .csv("/home/jovyan/data/costs_postgres.csv"))

In [16]:
df.show(5)

+----------+-----------+------+------+-----+
|      date|campaign_id| costs|clicks|views|
+----------+-----------+------+------+-----+
|2024-01-01|          1|670.52|    40|  110|
|2024-01-01|          2| 602.5|    11|  849|
|2024-01-01|          3|654.74|    51|  566|
|2024-01-01|          4|897.24|    86|  679|
|2024-01-01|          5|758.19|    30|  585|
+----------+-----------+------+------+-----+
only showing top 5 rows


In [21]:
df = (
    df
    .withColumn("date", F.to_date("date", "yyyy-MM-dd"))
    .withColumn("campaign_id", F.col("campaign_id").cast("int"))
    .withColumn("costs", F.col("costs").cast("double"))
    .withColumn("clicks", F.col("clicks").cast("int"))
    .withColumn("views", F.col("views").cast("int"))
)

df.write \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{ph_host}:{pg_port}/{pg_db}") \
    .option("dbtable", pg_table) \
    .option("user", pg_user) \
    .option("password", pg_pass) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [19]:
df.show(5)

+----------+-----------+------+------+-----+
|      date|campaign_id| costs|clicks|views|
+----------+-----------+------+------+-----+
|2024-01-01|          1|670.52|    40|  110|
|2024-01-01|          2| 602.5|    11|  849|
|2024-01-01|          3|654.74|    51|  566|
|2024-01-01|          4|897.24|    86|  679|
|2024-01-01|          5|758.19|    30|  585|
+----------+-----------+------+------+-----+
only showing top 5 rows


In [28]:
ch_host = 'clickhouse'
ch_port = '8123'
ch_db = 'default'
ch_table = 'visits'
ch_user = 'clickhouse'
ch_pass = 'clickhouse'

In [29]:
df = (spark.read
     .option('header', True)
     .csv("/home/jovyan/data/visits_clickhouse.csv"))

In [30]:
df = (
    df
    .withColumn("visitid", F.col("visitid").cast("int"))
    .withColumn("visitDateTime", F.to_timestamp("visitDateTime", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("duration", F.col("duration").cast("int"))
    .withColumn("clientID", F.col("clientID").cast("int"))
)

df.write \
    .format("jdbc") \
    .option("url", f"jdbc:clickhouse://{ch_host}:{ch_port}/{ch_db}") \
    .option("dbtable", ch_table) \
    .option("user", ch_user) \
    .option("password", ch_pass) \
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver") \
    .mode("append") \
    .save()

In [ ]:
# Создаем новый датафрейм для чтения из таблиц

In [48]:
visits = (
    spark.read
    .format("jdbc")
    .option("url", f"jdbc:clickhouse://{ch_host}:{ch_port}/{ch_db}")
    .option("dbtable", ch_table) 
    .option("user", ch_user) 
    .option("password", ch_pass) 
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
    .load()
)

costs = (
    spark.read
    .format("jdbc") 
    .option("url", f"jdbc:postgresql://{ph_host}:{pg_port}/{pg_db}") 
    .option("dbtable", pg_table) 
    .option("user", pg_user) 
    .option("password", pg_pass) 
    .option("driver", "org.postgresql.Driver") 
    .load()
)

In [54]:
filtered_step1 = (
    visits
    .withColumn('dt', F.date_format(F.col('visitDateTime'), 'yyyy-MM-dd'))
    .where(F.col('dt').between('2024-01-01', '2025-01-27'))
    .where(F.col('source').isin('ad', 'direct'))
    .where(F.col('URL').rlike('.*checkout.*|.*add.*|.*home.*|.*contact.*|.*top50.*|.*customer-service.*|.*wishlist.*|.*sale.*|.*best-sellers.*|.*view.*|.*discount.*|.*featured.*|.*new-arrivals.*|.*settings.*|.*return-policy.*|.*edit.*|.*delete.*|.*reviews.*|.*products.*|.*about.*'))
    .select(
        'dt',
        'visitid',
        'clientID',
        'URL',
        'duration',
        'source',
        'UTMCampaign',
        'params',
        F.regexp_replace(F.col('params'), r'\[|\]', '').alias('params_regex')
    )
    .withColumn('params_split', F.split('params_regex', ', '))
)


In [64]:
filtered_step1.show(3, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------
 dt           | 2024-08-31                                    
 visitid      | 100059                                        
 clientID     | 522                                           
 URL          | https://our-cool-website.com/featured         
 duration     | 79                                            
 source       | direct                                        
 UTMCampaign  | month_openconcept_kitchens_promotion_20240628 
 params       | []                                            
 params_regex |                                               
 params_split | []                                            
-RECORD 1-----------------------------------------------------
 dt           | 2024-04-03                                    
 visitid      | 100109                                        
 clientID     | 121                                           
 URL          | https://our-cool-website.com/view      

In [59]:
filtered_step2 = (
    filtered_step1
    .withColumn('event_type', F.regexp_replace(F.get(F.col('params_split'), 0), "'", ''))
    .withColumn('event_id', F.get(F.col('params_split'), 1).cast('int'))
)

In [60]:
filtered_step2.show(1, truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------
 dt           | 2024-08-31                                    
 visitid      | 100059                                        
 clientID     | 522                                           
 URL          | https://our-cool-website.com/featured         
 duration     | 79                                            
 source       | direct                                        
 UTMCampaign  | month_openconcept_kitchens_promotion_20240628 
 params       | []                                            
 params_regex |                                               
 params_split | []                                            
 event_type   |                                               
 event_id     | NULL                                          
only showing top 1 row


In [61]:
visits_df = (
    filtered_step2
    .where(F.col('event_type') == 'submit')
    .select(
        'dt',
        F.col('visitid').cast('string').alias('visitid'),
        F.col('clientID').cast('string').alias('clientid'),
        'URL',
        'duration',
        'source',
        'UTMCampaign',
        'event_type',
        'event_id'
    )
    .distinct()
)

In [62]:
visits_df.show(1, truncate=False, vertical=True)

-RECORD 0----------------------------------------------------
 dt          | 2024-07-16                                    
 visitid     | 294982                                        
 clientid    | 773                                           
 URL         | https://our-cool-website.com/customer-service 
 duration    | 57                                            
 source      | direct                                        
 UTMCampaign | year_traditional_kitchens_launch_20240707     
 event_type  | submit                                        
 event_id    | 9620                                          
only showing top 1 row


In [65]:
visits_df.count()

2433

In [66]:
costs_df = (
    costs
    .groupBy("date", "campaign_id")
    .agg(
        F.round(F.sum("costs"), 2).alias("costs"),
        F.sum("clicks").alias("clicks"),
        F.sum("views").alias("views")
    )
    .orderBy("date", "campaign_id")
)

In [67]:
costs_df.show(5)

+----------+-----------+------+------+-----+
|      date|campaign_id| costs|clicks|views|
+----------+-----------+------+------+-----+
|2024-01-01|          1|670.52|    40|  110|
|2024-01-01|          2|602.50|    11|  849|
|2024-01-01|          3|654.74|    51|  566|
|2024-01-01|          4|897.24|    86|  679|
|2024-01-01|          5|758.19|    30|  585|
+----------+-----------+------+------+-----+
only showing top 5 rows


In [68]:
costs_df.printSchema()

root
 |-- date: date (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- costs: decimal(23,2) (nullable = true)
 |-- clicks: long (nullable = true)
 |-- views: long (nullable = true)



In [74]:
campaigns_dict.createOrReplaceTempView('campaigns_dict_view')
campaigns_df = spark.sql("""
    select
        campaign_id,
        campaign_name,
        case
            when campaign_name like 'year%' then 'Год'
            when campaign_name like 'quarter%' then 'Квартал'
            when campaign_name like 'month%' then 'Месяц'
            else null
        end as campaign_duration
    from campaigns_dict_view
""")

In [75]:
campaigns_df.show(5)

+-----------+--------------------+-----------------+
|campaign_id|       campaign_name|campaign_duration|
+-----------+--------------------+-----------------+
|          1|year_modern_kitch...|              Год|
|          2|quarter_custom_ki...|          Квартал|
|          3|month_smart_kitch...|            Месяц|
|          4|year_luxury_kitch...|              Год|
|          5|quarter_ecofriend...|          Квартал|
+-----------+--------------------+-----------------+
only showing top 5 rows


In [78]:
submits = spark.read.parquet(f"{path}submits.parquet")

In [79]:
submits.show(5)

+---------+--------+-----------+
|submit_id|    name|      phone|
+---------+--------+-----------+
|     2282|Jennifer|79511904041|
|     9898| Jeffrey|79824419733|
|     9005|   Linda|79074725672|
|     1507|  Teresa|79864203598|
|     3803|   Tanya|79779567654|
+---------+--------+-----------+
only showing top 5 rows


In [81]:
submits_df = (
    submits.withColumn('phone', F.col('phone').cast('string'))
    .withColumn('phone_plus', F.concat(F.lit('+'), F.col('phone')))
    .withColumn('phone_md5', F.md5('phone'))
    .withColumn('phone_plus_md5', F.md5('phone_plus'))
)

In [82]:
submits_df.show(5)

+---------+--------+-----------+------------+--------------------+--------------------+
|submit_id|    name|      phone|  phone_plus|           phone_md5|      phone_plus_md5|
+---------+--------+-----------+------------+--------------------+--------------------+
|     2282|Jennifer|79511904041|+79511904041|4c7720fdf6f9eec62...|6ce81d5347bcd3ead...|
|     9898| Jeffrey|79824419733|+79824419733|9d106e45036bd4176...|43630233dcc965f98...|
|     9005|   Linda|79074725672|+79074725672|e962445cc1166c16a...|c165e91a7286aaf45...|
|     1507|  Teresa|79864203598|+79864203598|bb652c2e9fc072e10...|4f75f6da42c4bdfbb...|
|     3803|   Tanya|79779567654|+79779567654|9bbfc56351953a6a2...|58bc0c940162e3b23...|
+---------+--------+-----------+------------+--------------------+--------------------+
only showing top 5 rows


In [83]:
deals = spark.read.parquet(f'{path}deals.parquet')

In [85]:
deals.show(5)

+-------+----------+--------------------+-----------+--------------------+--------------------+
|deal_id| deal_date|                 fio|      phone|               email|             address|
+-------+----------+--------------------+-----------+--------------------+--------------------+
|      1|2024-03-04|          Gregory Wu|79746561889|  paul80@example.net|098 Yates Cliff A...|
|      2|2024-08-20|    William Ross Jr.|79074725672|  xyoung@example.org|197 Willie Groves...|
|      3|2024-10-15|          Sonya Kerr|79201244835|  elewis@example.com|144 Andrew Cape, ...|
|      4|2024-12-31|Mrs. Angela Tucke...|79771829751|robertparker@exam...|6056 Collins View...|
|      5|2024-03-23|         Eric Flores|79729054809|barbara73@example...|29684 Mcfarland B...|
+-------+----------+--------------------+-----------+--------------------+--------------------+
only showing top 5 rows


In [87]:
deals_df = (
    deals
    .withColumn('username', F.split(F.col('email'), '@').getItem(0))
    .withColumn('domain', F.split(F.col('email'), '@').getItem(1))
    .where(F.col('domain').isin('example.com', 'example.org', 'example.net'))
    .withColumn('phone', F.col('phone').cast('string'))
)

In [88]:
deals_df.show(2)

+-------+----------+----------------+-----------+------------------+--------------------+--------+-----------+
|deal_id| deal_date|             fio|      phone|             email|             address|username|     domain|
+-------+----------+----------------+-----------+------------------+--------------------+--------+-----------+
|      1|2024-03-04|      Gregory Wu|79746561889|paul80@example.net|098 Yates Cliff A...|  paul80|example.net|
|      2|2024-08-20|William Ross Jr.|79074725672|xyoung@example.org|197 Willie Groves...|  xyoung|example.org|
+-------+----------+----------------+-----------+------------------+--------------------+--------+-----------+
only showing top 2 rows


In [ ]:
#Собираем витрину

In [89]:
customer_detailed = (
    visits_df.alias('v')
    .join(
        submits_df.alias('s'), F.col('v.event_id') == F.col('s.submit_id'), 'left'
    )
    .join(
        deals_df.alias('d'), (F.col('s.phone') == F.col('d.phone')) &
        (F.col('v.dt') <= F.col('d.deal_date')), 'left'
    )
    .join(
        campaigns_df.alias('camp'), F.col('v.utmcampaign') == F.col('camp.campaign_id'), 'left'
    )
    .join(
        costs_df.alias('c'), (F.col('camp.campaign_id') == F.col('c.campaign_id')) &
        (F.col('v.dt') == F.col('c.date')), 'left'
    )
    .select(
        'v.dt',
        F.col('v.visitid').alias('visit_id'),
        F.col('v.clientid').alias('client_id'),
        'v.url',
        'v.duration',
        'v.source',
        'v.utmcampaign',
        'v.event_type',
        'v.event_id',
        's.submit_id',
        's.name',
        's.phone',
        's.phone_plus',
        's.phone_md5',
        's.phone_plus_md5',
        'd.deal_id',
        'd.deal_date',
        'd.fio',
        F.col('d.phone').alias('phone_deal'),
        'd.email',
        'd.address',
        'd.username',
        'd.domain',
        'camp.campaign_name',
        'camp.campaign_duration',
        'c.costs',
        'c.clicks',
        'c.views'
    )
)

In [96]:
customer_detailed.show(2, vertical = True)

-RECORD 0---------------------------------
 dt                | 2024-07-26           
 visit_id          | 504309               
 client_id         | 860                  
 url               | https://our-cool-... 
 duration          | 42                   
 source            | direct               
 utmcampaign       | quarter_custom_ki... 
 event_type        | submit               
 event_id          | 750                  
 submit_id         | 750                  
 name              | Scott                
 phone             | 79087701827          
 phone_plus        | +79087701827         
 phone_md5         | 7dff94d8480d3c518... 
 phone_plus_md5    | 93ec4e2988fa7f7b4... 
 deal_id           | NULL                 
 deal_date         | NULL                 
 fio               | NULL                 
 phone_deal        | NULL                 
 email             | NULL                 
 address           | NULL                 
 username          | NULL                 
 domain    

In [97]:
customer_detailed.count()

2639

In [99]:
customer_detailed.cache()

DataFrame[dt: string, visit_id: string, client_id: string, url: string, duration: int, source: string, utmcampaign: string, event_type: string, event_id: int, submit_id: bigint, name: string, phone: string, phone_plus: string, phone_md5: string, phone_plus_md5: string, deal_id: bigint, deal_date: string, fio: string, phone_deal: string, email: string, address: string, username: string, domain: string, campaign_name: string, campaign_duration: string, costs: decimal(23,2), clicks: bigint, views: bigint]

In [100]:
spark

In [101]:
campaigns_agg = (
    customer_detailed
    .groupBy('campaign_name')
    .agg(
        F.countDistinct('visit_id').alias('unique_visits'),
        F.countDistinct('client_id').alias('unique_clients'),
        F.countDistinct('submit_id').alias('unique_submits'),
        F.countDistinct('deal_id').alias('unique_deals'),
        F.sum('costs').alias('total_costs'),
        F.sum('clicks').alias('total_clicks'),
        F.sum('views').alias('total_views'),
        F.sum('duration').alias('total_duration')
    )
    .withColumn('avg_deal_cost', (F.col('total_costs') / F.col('unique_deals')).cast('decimal(19,2)'))
)

In [102]:
campaigns_agg.cache().count()

1

In [103]:
dates_agg = (
    customer_detailed
    .groupBy(F.substring('dt', 1, 7).alias('month'))  # 2025-01-01
    .agg(
        F.countDistinct('visit_id').alias('unique_visits'),
        F.countDistinct('client_id').alias('unique_clients'),
        F.countDistinct('submit_id').alias('unique_submits'),
        F.countDistinct('deal_id').alias('unique_deals'),
        F.sum('costs').alias('total_costs'),
        F.sum('clicks').alias('total_clicks'),
        F.sum('views').alias('total_views'),
        F.sum('duration').alias('total_duration')
    )
    .withColumn('avg_deal_cost', (F.col('total_costs') / F.col('unique_deals')).cast('decimal(19,2)'))
)

In [104]:
dates_agg.cache().count()

13

In [111]:
def save_to_postgres(df, table_name):
    (
        df.write
        .format('jdbc')
        .option('url', f'jdbc:postgresql://{ph_host}:{pg_port}/{pg_db}')
        .option('dbtable', table_name)
        .option('user', pg_user)
        .option('password', pg_passsave_to_postgres(campaigns_agg, 'campaigns_agg')
)
        .option('driver', 'org.postgresql.Driver')
        .mode("overwrite")
        .save()
    )

In [112]:
save_to_postgres(customer_detailed, 'customer_detailed')

In [113]:
save_to_postgres(campaigns_agg, 'campaigns_agg')


In [114]:
save_to_postgres(dates_agg, 'dates_agg')

In [115]:
dates_agg.unpersist()

DataFrame[month: string, unique_visits: bigint, unique_clients: bigint, unique_submits: bigint, unique_deals: bigint, total_costs: decimal(33,2), total_clicks: bigint, total_views: bigint, total_duration: bigint, avg_deal_cost: decimal(19,2)]

In [116]:
campaigns_agg.unpersist()

DataFrame[campaign_name: string, unique_visits: bigint, unique_clients: bigint, unique_submits: bigint, unique_deals: bigint, total_costs: decimal(33,2), total_clicks: bigint, total_views: bigint, total_duration: bigint, avg_deal_cost: decimal(19,2)]

In [117]:
spark.stop()